# 02 Structure of the feasible sets

This notebook computes the optimal solution of the 200-day sample average approximation (SAA) for every setting: the feasible plans, the SAA front in the plane of labor and waiting cost, and the cheapest feasible plan. It also computes the numbers on the structure of the feasible sets quoted in Appendix A (minimal plans and the product factor) and tests whether adding a staff member can lower the simulated service level.

**Needs:** the main-sample enumerations from notebook 01. Finished settings are read from `results/parts/` and not computed again.

**Outputs**

| File | Use in the paper |
|---|---|
| `tables/structure.tex` | Table A.3, optimal solution of the 200-day SAA |
| `results/structure.csv`, `results/monotonicity.csv`, `results/fronts.csv` | Numbers quoted in Appendix A; fronts reused by the other notebooks |

**Run time:** a few minutes per three-block setting the first time, a few seconds once the parts exist.

In [1]:
# Setup: storage folder, numba and the core module written by notebook 00
import os, sys, shutil, subprocess, time, math
import numpy as np
import pandas as pd

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/My Drive/clinic2026_final'
else:
    BASE = os.environ.get('CLINIC_BASE', os.path.abspath('clinic2026_final'))
try:
    import numba  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numba'], check=True)

CORE = os.path.join(BASE, 'core', 'clinic_core.py')
if not os.path.exists(CORE):
    raise FileNotFoundError('Run notebook 00 first; it writes the core module to ' + CORE)
LOCAL = '/content/clinic_core_local' if IN_COLAB else os.path.join(BASE, 'core')
os.makedirs(LOCAL, exist_ok=True)
if os.path.abspath(LOCAL) != os.path.abspath(os.path.dirname(CORE)):
    shutil.copy(CORE, LOCAL)          # numba caches its compiled code next to the module, so keep it on local disk
sys.path.insert(0, LOCAL)
import clinic_core as cc
P = cc.make_paths(BASE)
os.makedirs(os.path.join(P['results'], 'parts'), exist_ok=True)
print('Project folder:', BASE)
print('numba available:', cc.HAVE_NUMBA)

Mounted at /content/drive
Project folder: /content/drive/My Drive/clinic2026_final
numba available: True


In [2]:
def part_file(kind, s):
    return os.path.join(P['results'], 'parts', f'{kind}__{s.name}.csv')


def have_parts(kinds, s):
    return all(os.path.exists(part_file(k, s)) for k in kinds)


def available(s, n_days=200, seed_offset=0):
    return os.path.exists(cc.enum_file(P['data'], s, n_days, seed_offset))


def combine(kind):
    files = [part_file(kind, s) for s in cc.grid() if os.path.exists(part_file(kind, s))]
    frames = [pd.read_csv(f) for f in files if os.path.getsize(f) > 1]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

In [3]:
# Structure and monotonicity per setting (resumable: finished settings are skipped)
MAX_MINUTES = 50
t_start = time.time()
for s in cc.grid():
    if have_parts(('structure', 'monotonicity', 'front'), s):
        continue
    if not available(s):
        print(f'{s.name}: enumeration not available yet; skipped')
        continue
    if (time.time() - t_start) / 60 > MAX_MINUTES:
        print('Time budget reached; run this cell again to continue.')
        break
    t0 = time.time()
    df = cc.load_table(P['data'], s)
    row, mins, front = cc.structure(df, s)
    ev = cc.Evaluator(s, *cc.MAIN)
    mono = cc.monotonicity(df, s, ev)
    fr = front.assign(**cc.tag(s), plan=[cc.staffing_text(cc.plan_of_row(i, s), s) for i in front.index])
    pd.DataFrame([row]).to_csv(part_file('structure', s), index=False)
    pd.DataFrame([mono]).to_csv(part_file('monotonicity', s), index=False)
    fr.to_csv(part_file('front', s), index=False)
    print(f'{s.name}: {row["feasible"]:,} feasible, {row["minimal"]} minimal, product factor '
          f'{row["product factor"]:.2f}, front {row["front"]}, {time.time() - t0:.0f} s')
structure = combine('structure')
mono = combine('monotonicity')
fronts = pd.concat([pd.read_csv(part_file('front', s)) for s in cc.grid() if os.path.exists(part_file('front', s))],
                   ignore_index=True) if len(structure) else pd.DataFrame()
if len(structure):
    structure.to_csv(os.path.join(P['results'], 'structure.csv'), index=False)
    mono.to_csv(os.path.join(P['results'], 'monotonicity.csv'), index=False)
    fronts.to_csv(os.path.join(P['results'], 'fronts.csv'), index=False)
print(f'{len(structure)} of {len(cc.grid())} settings done')

B2-PAR1.0-moderate: 303 feasible, 19 minimal, product factor 1.58, front 22, 12 s
B2-PAR1.0-high: 174 feasible, 17 minimal, product factor 1.81, front 15, 6 s
B2-PAR1.3-moderate: 225 feasible, 12 minimal, product factor 1.69, front 19, 6 s
B2-PAR1.3-high: 96 feasible, 13 minimal, product factor 1.95, front 11, 5 s
B2-PAR1.6-moderate: 126 feasible, 9 minimal, product factor 1.21, front 13, 8 s
B2-PAR1.6-high: 68 feasible, 5 minimal, product factor 1.41, front 10, 8 s
B2-PAR2.0-moderate: 40 feasible, 4 minimal, product factor 1.50, front 11, 7 s
B2-PAR2.0-high: 15 feasible, 4 minimal, product factor 1.07, front 6, 5 s
B3-PAR1.0-moderate: 8,509 feasible, 227 minimal, product factor 3.30, front 32, 11 s
B3-PAR1.0-high: 3,748 feasible, 223 minimal, product factor 3.33, front 31, 11 s
B3-PAR1.3-moderate: 7,045 feasible, 219 minimal, product factor 2.94, front 42, 12 s
B3-PAR1.3-high: 2,823 feasible, 204 minimal, product factor 5.27, front 37, 12 s
B3-PAR1.6-moderate: 3,925 feasible, 173 mini

In [4]:
# Helpers that write the tables in the exact format of the paper
EPS = 1e-9


def order(d):
    return d.assign(_cv=d.cv.map({'moderate': 0, 'high': 1})).sort_values(['blocks', 'PAR', '_cv'])


def table_tex(colspec, head, body, cap, label, box=r'max width=\textwidth, max totalheight=0.88\textheight'):
    """A table in adjustbox with a pipe-delimited tabular, as in the manuscript."""
    return '\n'.join([r'\begin{table}[htbp]', r'\centering', rf'\begin{{adjustbox}}{{{box}}}',
                      rf'\begin{{tabular}}{{{colspec}}}', r'\hline', head, r'\hline'] + list(body)
                     + [r'\end{tabular}', r'\end{adjustbox}', rf'\caption{{{cap}}}', rf'\label{{{label}}}',
                        r'\end{table}']) + '\n'


def write_table(name, text):
    path = os.path.join(P['tables'], name)
    cc.write_text(path, text)
    print('Written', path)

In [5]:
# Table A.3: optimal solution of the 200-day SAA
st = order(structure)
body = []
for B, g in st.groupby('blocks', sort=True):
    body += [rf'\multicolumn{{7}}{{|l|}}{{{B} blocks of {480 // B} minutes, {int(g.staffings.iloc[0]):,} plans}} \\', r'\hline']
    for _, r in g.iterrows():
        body.append(f"{int(r.blocks)} & {r.PAR:.1f} & {r.cv} & {int(r.feasible):,} ({100 * r['feasible share']:.2f}) & "
                    f"{int(r.front)} & {r['cheapest M']:,.0f} & {r['cheapest plan']} \\\\")
    body.append(r'\hline')
head = (r'Blocks & $\pi$ & CV & \makecell{Feasible\\plans (\%)} & \makecell{Plans on the\\SAA front} & '
        r'\makecell{Cheapest\\cost (\$)} & Cheapest feasible plan \\')
cap = (r'Optimal solution of the 200-day SAA. Plans list receptionists, nurses, physicians and pharmacists in each block. '
       r'CV: service-time variability.')
write_table('structure.tex', table_tex('|c|c|c|r|c|r|l|', head, body, cap, 'tab:structure'))

Written /content/drive/My Drive/clinic2026_final/tables/structure.tex


In [6]:
# Numbers quoted in Appendix A
for B, g in structure.groupby('blocks'):
    print(f'{B} blocks: feasible share {100 * g["feasible share"].min():.2f}% to {100 * g["feasible share"].max():.2f}%; '
          f'plans from which no staff member can be removed without failing the standard (minimal plans) '
          f'{g.minimal.min()} to {g.minimal.max()}; product factor {g["product factor"].min():.2f} to '
          f'{g["product factor"].max():.2f}; plans on the SAA front {g.front.min()} to {g.front.max()}')
for B, g in mono.groupby('blocks'):
    share = g['SL decreases'] / g['pairs']
    print(f'{B} blocks, adding one staff member: the simulated service level decreases in {100 * share.min():.1f}% to '
          f'{100 * share.max():.1f}% of the pairs, by at most {g["largest SL decrease"].max():.4f}; significant after '
          f'Bonferroni in {int((g["significant Bonferroni"] > 0).sum())} settings (highest service level among them '
          f'{g["highest SL among significant"].max():.2f}); a feasible plan becomes infeasible in {int(g.crossings.sum())} '
          f'pairs (largest drop {g["largest crossing drop"].max():.4f}, significant {int(g["crossings significant Bonferroni"].sum())})')
mono[['setting', 'SL decreases', 'largest SL decrease', 'significant Bonferroni', 'significant where',
      'highest SL among significant', 'crossings', 'largest crossing drop', 'crossings significant Bonferroni']]

2 blocks: feasible share 0.23% to 4.62%; plans from which no staff member can be removed without failing the standard (minimal plans) 4 to 19; product factor 1.07 to 1.95; plans on the SAA front 6 to 22
3 blocks: feasible share 0.05% to 1.60%; plans from which no staff member can be removed without failing the standard (minimal plans) 34 to 227; product factor 2.94 to 5.98; plans on the SAA front 17 to 42
2 blocks, adding one staff member: the simulated service level decreases in 1.6% to 3.9% of the pairs, by at most 0.0074; significant after Bonferroni in 0 settings (highest service level among them nan); a feasible plan becomes infeasible in 0 pairs (largest drop nan, significant 0)
3 blocks, adding one staff member: the simulated service level decreases in 2.4% to 4.9% of the pairs, by at most 0.0392; significant after Bonferroni in 6 settings (highest service level among them 0.44); a feasible plan becomes infeasible in 3 pairs (largest drop 0.0006, significant 0)


,setting,SL decreases,largest SL decrease,significant Bonferroni,significant where,highest SL among significant,crossings,largest crossing drop,crossings significant Bonferroni
0,B2-PAR1.0-moderate,904,0.001989,0,NaN,NaN,0,NaN,0
1,B2-PAR1.0-high,650,0.002355,0,NaN,NaN,0,NaN,0
2,B2-PAR1.3-moderate,735,0.003949,0,NaN,NaN,0,NaN,0
3,B2-PAR1.3-high,548,0.002169,0,NaN,NaN,0,NaN,0
4,B2-PAR1.6-moderate,999,0.001682,0,NaN,NaN,0,NaN,0
5,B2-PAR1.6-high,899,0.003278,0,NaN,NaN,0,NaN,0
6,B2-PAR2.0-moderate,1359,0.005443,0,NaN,NaN,0,NaN,0
7,B2-PAR2.0-high,580,0.007385,0,NaN,NaN,0,NaN,0
8,B3-PAR1.0-moderate,132995,0.019708,0,NaN,NaN,2,0.000587,0
9,B3-PAR1.0-high,105737,0.018535,0,NaN,NaN,0,NaN,0
